# Sudan war displacement mapping — how violence and displacement co-evolved

*Two years into the Sudan war (April 2023 onward), how have violence and
displacement co-evolved geographically across Sudan and its neighbours?*

## The question

In April 2023, fighting between the Sudanese Armed Forces and the Rapid Support
Forces broke out in Khartoum and spread across the country. It became one of the
largest displacement crises of the decade — and one of the least-covered.

This notebook builds a **descriptive, geographic-temporal** picture of how
**violence** and **displacement** co-evolved across Sudan's 18 states (and its
five neighbours, as refugee destinations) from April 2023 onward. It is
cartographic by design: we describe *where* and *when*. We do **not** attribute
displacement causally to specific events or actors, estimate humanitarian needs,
or forecast future flows — each is a separate methodology.

This first part builds the **violence layer** from ACLED event data. The
displacement layer (IOM DTM + UNHCR) and the maps follow in later sections.

## Data

| Source | Role | Granularity | Window | Caveats |
|--------|------|-------------|--------|---------|
| [ACLED](https://acleddata.com/) | Violence layer | Event-level (date, lat/lon, type) | Apr 2023 – May 2025 | Media-coded — well-covered areas over-represented. Read from a pinned snapshot. |
| [IOM DTM](https://dtm.iom.int/sudan) | Internal-displacement layer | Admin-1 present-IDP stock | Aug 2023 – Feb 2026 | Field-assessed; irregular round cadence. Three overlapping operations (Decision 7). Pinned snapshot. |
| [UNHCR](https://data.unhcr.org/en/situations/sudansituation) | Cross-border-displacement layer | Destination country | 2023 – 2026 | Two sources disagree by design (Decision 5); per-country snapshot dates differ. |
| GADM 4.1 + admin-1 crosswalk | Geo basemap | Admin-1 polygons / pcodes | — | Built in the geo-reconciliation step (Decision 6). |

The ACLED API is gated behind a Research tier this account does not hold, so the
analytic input is a **manual Data Export Tool snapshot**, pinned to
`data/processed/acled_snapshot_2026-05-19.parquet` (27,725 events across six
countries). ACLED revises history as new sourcing emerges — every run reads the
pinned snapshot, never the live API. The IOM DTM admin-1 IDP series is likewise
pinned (`dtm_admin1_snapshot_2026-05-19.parquet`); UNHCR figures are pulled live
through an HTTP cache (small, key-free).

**Window — April 2023 to May 2025.** The account's tier withholds the most
recent ~12 months of events, so a snapshot taken on 2026-05-19 ends at
2025-05-19. This is the genuine maximum available, not a chosen cut-off; it
happens to align with the "two years into the war" framing.


In [1]:
import warnings

import pandas as pd

from sudan_displacement import data
from sudan_displacement.diagnostics import compare_alternatives

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_rows", 60)

# The three kinetic event types that the violence layer keeps (see Decision 1).
VIOLENT_TYPES = list(data.VIOLENT_EVENT_TYPES)

# The pinned ACLED snapshot — the canonical analytic input (never the live API).
acled = pd.read_parquet(data._latest_snapshot("acled_snapshot"))
sudan = acled[acled["country"] == "Sudan"].copy()
sudan["event_date"] = pd.to_datetime(sudan["event_date"])

print(f"ACLED snapshot : {len(acled):,} events, {acled['country'].nunique()} countries")
print(f"Sudan subset   : {len(sudan):,} events, "
      f"{sudan['event_date'].min().date()} -> {sudan['event_date'].max().date()}")

ACLED snapshot : 27,725 events, 6 countries
Sudan subset   : 16,569 events, 2023-04-01 -> 2025-05-19


## Cleaning & structuring

Four data-processing decisions turn raw ACLED events into the analytic violence
layer. Each is documented in five parts — **problem, diagnostic, options,
decision + rationale, sensitivity** — per the project's decision discipline.

| # | Decision | Question |
|---|----------|----------|
| 6 | Admin-1 reconciliation | How do we co-register ACLED, DTM and GADM admin-1 units? |
| 1 | Event-type filter | Which ACLED event types count as "violence"? |
| 2 | Geo-precision threshold | Do we drop imprecisely geocoded events? |
| 3 | Temporal aggregation | Weekly or monthly time bins? |

Decision 6 comes first because the other three operate on events already joined
to a common admin-1 grid.

### Decision 6: Admin-1 boundary reconciliation — one crosswalk for three naming systems

**Problem.** The violence layer (ACLED), the internal-displacement layer (IOM
DTM) and the polygon basemap (GADM 4.1) each label admin-1 units with a
different convention. ACLED uses spaced English names (`North Kordofan`,
`Al Jazirah`); DTM uses OCHA names plus pcodes (`Aj Jazirah`, `SD01`–`SD18`);
GADM uses BGN/PCGN romanisation with no spaces (`NorthKurdufan`, `AlQadarif`).
Nothing downstream works until these agree: the violence and displacement layers
cannot be joined to the same polygon, and neither can be drawn on the
choropleth. A careless join fails *silently* — ACLED's `Gedaref` simply never
matching GADM's `AlQadarif` would zero out an entire state rather than raise an
error.

**Diagnostic.** Exact-string match rate against GADM `NAME_1`, then the rate
after normalisation (strip accents, spacing, casing; bridge the known
`Kurdufan`↔`Kordofan` and `Aj`↔`Al Jazirah` token splits).

In [2]:
crosswalk = data.load_admin1_crosswalk()
crosswalk.groupby(["country", "match_method"]).size().unstack(fill_value=0)

match_method,exact,manual_override,normalized,unmatched
country,,,,
CentralAfricanRepublic,11,0,6,0
Chad,10,1,10,2
Egypt,2,0,0,25
Ethiopia,5,4,1,1
SouthSudan,2,5,3,0
Sudan,4,1,13,0


The diagnostic surfaced four things:

- **Sudan:** 4/18 match exactly, 17/18 after normalisation. Two real gaps —
  `Gedaref`/`AlQadarif` (genuinely different romanisations, no shared normalised
  form) and **Abyei**, which ACLED reports as a 19th Sudan admin-1 unit but which
  has no GADM polygon and no DTM pcode (it is a disputed Sudan/South Sudan
  territory).
- **Neighbours:** Central African Republic 17/17 and Chad 21/21 reconcile on
  normalisation alone; South Sudan needs a handful of romanisation overrides;
  **Ethiopia** reconciles only 10/14 because GADM 4.1 predates Ethiopia's
  2020–2023 regional reorganisation; **Egypt** reconciles 2/27 — ACLED and GADM
  use entirely incompatible governorate romanisations.
- **GADM `HASC_1` is unusable as a key:** North Kurdufan and West Kurdufan both
  carry `SD.KN`. `GID_1` is the only unique GADM key.
- Abyei is **181 ACLED events (1.09%) and 515 fatalities (1.15%)** of Sudan's
  totals — small, but not nothing.

**Options considered.**
- (a) **Exact string join.** Rejected — only 4/18 in Sudan; would silently void
  14 states.
- (b) **Fuzzy / edit-distance matching.** Rejected — opaque, and unsafe here:
  `North Kordofan` and `South Kordofan` differ by a single token, so a threshold
  loose enough to catch real variants is also loose enough to cross-wire
  neighbouring states.
- (c) **Normalisation + a small hand-audited override table, keyed on OCHA
  pcodes.** Chosen.

**Decision.** Build a deterministic crosswalk (`admin1_crosswalk.csv`, one row
per GADM polygon): normalise names, then resolve the few irreducible
romanisation gaps with an explicit, auditable override table. The canonical key
is the **OCHA pcode `SD01`–`SD18`** for Sudan — it is also DTM's key and the
humanitarian-sector standard — and GADM `GID_1` for the neighbours. **Abyei**'s
ACLED events are folded into **South Kordofan**, the Sudan state Abyei was
administratively carved out of and the one it most directly adjoins. **Egypt**'s
admin-1 units are deliberately left unreconciled: Egypt's role here is a refugee
*destination*, joined at country level, and no deliverable uses its admin-1
violence detail. Result: **Sudan reconciles 18/18**, every state carrying a
pcode, an ACLED name and a DTM name.

**Sensitivity.** The headline figures combine violence and displacement on
Sudan's 18 states, and that reconciliation is *exact* (18/18) — the main finding
rests on no judgement call in the join itself. The one judgement is the
Abyei → South Kordofan merge; at 1.1% of Sudan events its effect on any state's
counts is marginal, and it is re-run against the drop-Abyei alternative in
`03_robustness.ipynb`. Ethiopia's 4 unmatched regions and Egypt's unreconciled
governorates touch only neighbour-side admin-1 detail, which never enters a
headline figure.

### Decision 1: Event-type filtering — what counts as "violence"

**Problem.** ACLED codes every event into one of six `event_type` values across
three `disorder_type` families. Some are unambiguous armed violence; others —
peaceful protests, or "strategic developments" like negotiated agreements and
non-violent territory transfers — are not. Including the wrong categories
inflates the violence layer in exactly the wrong places: a state hosting
ceasefire talks would light up as violent. The choice changes the event count
per state materially, so it must be explicit.

**Diagnostic.** Cross-tabulate `event_type` against `disorder_type`, check how
fatalities concentrate across types — a genuine kinetic-violence type should
carry the deaths — and compare the total event count under each candidate
filter.

In [3]:
xtab = (sudan.groupby(["disorder_type", "event_type"]).size()
        .rename("n_events").to_frame())

per_type = (
    sudan.groupby("event_type")
    .agg(n_events=("event_id_cnty", "size"), fatalities=("fatalities", "sum"))
    .assign(
        pct_events=lambda d: (d["n_events"] / d["n_events"].sum() * 100).round(1),
        pct_fatalities=lambda d: (d["fatalities"] / d["fatalities"].sum() * 100).round(2),
    )
    .sort_values("fatalities", ascending=False)
)
display(xtab)
display(per_type)

compare_alternatives(
    sudan,
    {
        "(a) Battles + VAC": lambda d: d[d.event_type.isin(
            ["Battles", "Violence against civilians"])],
        "(b) + Explosions/Remote": lambda d: d[d.event_type.isin(VIOLENT_TYPES)],
        "(c) all event types": lambda d: d,
    },
)

n_events
disorder_type                      event_type                          
Demonstrations                     Protests                         324
                                   Riots                             16
Political violence                 Battles                         5275
                                   Explosions/Remote violence      3930
                                   Riots                             18
                                   Violence against civilians      3482
Political violence; Demonstrations Protests                           7
Strategic developments             Strategic developments          3517

,n_events,fatalities,pct_events,pct_fatalities
event_type,,,,
Battles,5275,28581,31.8,63.88
Explosions/Remote violence,3930,8071,23.7,18.04
Violence against civilians,3482,8057,21.0,18.01
Riots,34,16,0.2,0.04
Strategic developments,3517,12,21.2,0.03
Protests,331,6,2.0,0.01


,alternative,n_rows,n_cols
0,(a) Battles + VAC,8757,35
1,(b) + Explosions/Remote,12687,35
2,(c) all event types,16569,35


**Options considered.**
- (a) **Battles + Violence against civilians only** — the narrowest "armed
  conflict" reading; drops `Explosions/Remote violence` (air/drone strikes,
  shelling), which is perverse for a war defined by aerial bombardment.
- (b) **Battles + Explosions/Remote violence + Violence against civilians** —
  all three kinetic `Political violence` types.
- (c) **All event types, weighted** — keep protests and strategic developments
  at a lower weight; opaque, and mixes non-violence into a "violence" measure.

**Decision.** Option (b): the violence layer is **Battles + Explosions/Remote
violence + Violence against civilians**. The diagnostic anchors this — these
three carry **99.9%** of Sudan's fatalities (44,709 of 44,715), while `Protests`,
`Riots` and `Strategic developments` together account for 34 deaths.
`Strategic developments` is non-kinetic by ACLED's own definition (agreements,
non-violent transfers, recruitment) and `Demonstrations` (protests/riots) is a
distinct phenomenon from armed war violence. Option (a) is rejected because
excluding air/drone strikes and shelling would discard 3,930 events in a war
where remote violence is central.

**Sensitivity.** The drop-Explosions alternative (a) is re-run in
`03_robustness.ipynb`; it removes ~31% of violent events but, being concentrated
in the same heavily contested states, leaves the geographic ranking essentially
unchanged. The headline maps are not sensitive to including vs excluding the
~0.3% non-kinetic tail.

### Decision 2: Geocoding precision threshold

**Problem.** ACLED's `geo_precision` flags how confident the geocoding is:
1 = an exact town/site, 2 = near a known place, 3 = only the wider region is
known so the coordinates are an admin-area centroid. Conventionally an analysis
drops imprecise events. But this analysis aggregates to **admin-1** — and the
risk runs the other way: dropping precision-3 events would discard events that
*are* correctly placed at admin-1, and `geo_precision` degrades most in remote,
conflict-active areas (`CLAUDE.md`), so a strict threshold would bias the map
against exactly the rural states the project exists to describe.

**Diagnostic.** Tabulate `geo_precision`, and — the load-bearing check — confirm
the `admin1` label is populated even at the coarsest precision tier.

In [4]:
prec = (
    sudan.groupby("geo_precision")
    .agg(n_events=("event_id_cnty", "size"),
         admin1_missing=("admin1", lambda s: int(s.isna().sum())))
    .assign(pct_events=lambda d: (d["n_events"] / d["n_events"].sum() * 100).round(1))
)
display(prec)

# What would each candidate threshold drop?
compare_alternatives(
    sudan,
    {
        "(c) keep all":       lambda d: d,
        "(a) drop prec >= 3": lambda d: d[d["geo_precision"] < 3],
        "drop prec >= 2":     lambda d: d[d["geo_precision"] < 2],
    },
)

,n_events,admin1_missing,pct_events
geo_precision,,,
1,13433,0,81.1
2,2567,0,15.5
3,569,0,3.4


,alternative,n_rows,n_cols
0,(c) keep all,16569,35
1,(a) drop prec >= 3,16000,35
2,drop prec >= 2,13433,35


**Options considered.**
- (a) **Drop `geo_precision` ≥ 3** (strict) — the conventional cleaning step.
- (b) **Drop `geo_precision` ≥ 4** (loose) — moot here: the snapshot contains no
  tier-4 events (precision tops out at 3).
- (c) **Keep all tiers.**

**Decision.** Option (c): **keep all events**. The diagnostic is decisive — the
`admin1` field is populated for **100%** of Sudan events at *every* precision
tier, including all 569 tier-3 events. Since the analysis resolves to admin-1,
tier-3 precision (an admin-area centroid) is still exactly precise enough; the
coordinate imprecision it flags only matters for sub-admin-1 placement, which no
deliverable uses. Dropping tier 3 would remove 3.4% of events for no gain in the
admin-1 layer, while biasing against remote states. Option (a) is the
convention; the diagnostic shows the convention does not apply at this
aggregation level.

**Sensitivity.** `03_robustness.ipynb` re-runs the layer under the strict
drop-≥3 threshold; because tier-3 events are only 3.4% of the total and spread
across states, no state's rank changes. This is close to a definitional
non-issue at admin-1 resolution.

### Decision 3: Temporal aggregation — weekly or monthly bins

**Problem.** The heatmap and the animated dashboard need a time bin. Weekly bins
are responsive — they catch a short burst of fighting — but noisy. Monthly bins
are smoother and read cleanly on a heatmap, but can mask a sharp two-week spike.
With 18 states of very uneven activity, the wrong choice either floods the quiet
states with empty cells (weekly) or blurs the volatile ones (monthly).

**Diagnostic.** A signal-to-noise check on five held-out states spanning the
activity range (Khartoum, busiest; Kassala / Northern, quietest). For each, at
both frequencies: **lag-1 autocorrelation** (signal persistence — higher means
more signal, less noise) and the share of **empty bins** (a structural-sparsity
symptom).

In [5]:
viol = sudan[sudan["event_type"].isin(VIOLENT_TYPES)]


def signal_to_noise(sub, freq):
    s = sub.set_index("event_date").assign(n=1)["n"].resample(freq).sum().astype(float)
    return {"n_bins": len(s), "mean_per_bin": round(s.mean(), 1),
            "lag1_autocorr": round(s.autocorr(lag=1), 3),
            "pct_empty_bins": round((s == 0).mean() * 100, 1)}


rows = []
for a1 in ["Khartoum", "North Darfur", "White Nile", "Kassala", "Northern"]:
    sub = viol[viol["admin1"] == a1]
    for freq, label in [("W", "weekly"), ("MS", "monthly")]:
        rows.append({"admin1": a1, "n_events": len(sub), "bin": label,
                     **signal_to_noise(sub, freq)})
pd.DataFrame(rows)

,admin1,n_events,bin,n_bins,mean_per_bin,lag1_autocorr,pct_empty_bins
0,Khartoum,6080,weekly,111,54.8,0.795,0.0
1,Khartoum,6080,monthly,26,233.8,0.795,0.0
2,North Darfur,1582,weekly,113,14.0,0.693,4.4
3,North Darfur,1582,monthly,26,60.8,0.755,0.0
4,White Nile,285,weekly,111,2.6,0.280,35.1
5,White Nile,285,monthly,26,11.0,0.269,15.4
6,Kassala,32,weekly,111,0.3,0.372,82.9
7,Kassala,32,monthly,26,1.2,0.253,46.2
8,Northern,58,weekly,110,0.5,0.412,74.5
9,Northern,58,monthly,26,2.2,0.467,26.9


**Options considered.**
- (a) **Weekly bins** (~111 bins) — most responsive.
- (b) **Monthly bins** (26 bins) — smoother, fewer frames.
- (c) **Adaptive** (weekly for busy states, monthly for quiet) — inconsistent
  axes across the small-multiple heatmap; rejected as unreadable.

**Decision.** Option (b): **monthly bins**. The diagnostic shows weekly buys no
signal — lag-1 autocorrelation is *identical* for Khartoum (0.80 weekly = 0.80
monthly) and actually *improves* at monthly for North Darfur (0.69 → 0.76). What
weekly does buy is sparsity: in the quiet states it leaves **35–83%** of bins
empty (White Nile 35%, Northern 75%, Kassala 83%), which on a heatmap is an
ambiguous wall of blank cells and makes the animation flicker. Monthly cuts that
to 0–46%. With 26 monthly bins over the 25-month window the animation still has
ample frames.

**Sensitivity.** `build_violence_layer(freq="W")` regenerates the whole layer
weekly with a single argument; `03_robustness.ipynb` renders the heatmap both
ways. The geographic ranking of states is invariant to the bin width — only the
temporal smoothness of the animation changes.

### Building the violence layer

The four decisions above are encoded as the defaults of
`build_violence_layer()`: it filters to the three kinetic event types (D1),
keeps all precision tiers (D2), folds Abyei into South Kordofan and joins the
crosswalk (D6), and bins monthly (D3). The result is **grid-complete** — every
(state × month) cell exists, zero-filled — so the heatmap and animation never
confuse "no violence" with "no data".

In [6]:
violence = data.build_violence_layer()  # D1+D2+D3+D6 defaults; pins the parquet

n_states = violence["canonical_pcode"].nunique()
n_months = violence["period"].nunique()
print(f"Violence layer : {violence.shape[0]} rows = {n_states} states x {n_months} months")
print(f"Raw Sudan      : {len(sudan):,} events  ->  violent only: {len(viol):,} events")
print(f"Layer totals   : {violence['n_events'].sum():,} violent events, "
      f"{violence['fatalities'].sum():,} fatalities")
print(f"Empty cells    : {(violence['n_events'] == 0).sum()} / {len(violence)} "
      f"(genuine zero-violence state-months)")
print(f"Pinned to      : data/processed/violence_admin1_monthly.parquet")
violence.head()

Violence layer : 468 rows = 18 states x 26 months
Raw Sudan      : 16,569 events  ->  violent only: 12,687 events
Layer totals   : 12,687 violent events, 44,709 fatalities
Empty cells    : 75 / 468 (genuine zero-violence state-months)
Pinned to      : data/processed/violence_admin1_monthly.parquet


,canonical_pcode,admin1,period,n_events,fatalities
0,SD01,Khartoum,2023-04-01,206,330
1,SD01,Khartoum,2023-05-01,306,321
2,SD01,Khartoum,2023-06-01,265,540
3,SD01,Khartoum,2023-07-01,366,881
4,SD01,Khartoum,2023-08-01,495,1666


In [7]:
# State totals over the full window — the geographic shape of the war.
(violence.groupby("admin1")
 .agg(events=("n_events", "sum"), fatalities=("fatalities", "sum"))
 .sort_values("events", ascending=False))

,events,fatalities
admin1,,
Khartoum,6080,11264
North Darfur,1582,11793
Al Jazirah,1571,4983
North Kordofan,564,2312
South Darfur,543,2595
South Kordofan,495,1992
Sennar,309,1223
West Darfur,303,5142
White Nile,285,789


The violence layer is built and pinned to
`data/processed/violence_admin1_monthly.parquet` — 468 rows (18 states ×
26 months), 12,687 events, 44,709 fatalities. Khartoum dominates (6,080 events),
followed by North Darfur and Al Jazirah; the eastern states (Kassala, Red Sea)
are near-quiet. This is the first of the two analytic layers the maps need.

> **Note — the final month is partial.** The May 2025 bin runs only to the
> 19th (the snapshot's last date). It is kept so the window matches the stated
> Apr 2023 – May 2025 span, but the heatmap and animation should be read with
> that truncation in mind; it is recorded in the limitations.

**Next:** the displacement layer — IOM DTM internal displacement and UNHCR
cross-border flows — reconciled and co-registered on the same admin-1 grid
(Decision 5).

## The displacement layer

The violence layer answers *where the war was fought*. The displacement layer
answers *where it pushed people* — and that splits into two physically distinct
flows that no single source measures together:

- **Internal displacement** — Sudanese uprooted but still inside Sudan. Measured
  by **IOM's Displacement Tracking Matrix (DTM)** through field assessments,
  reported as a present-IDP *stock* per admin-1 state.
- **Cross-border displacement** — Sudanese who fled the country. Measured by
  **UNHCR**, reported per destination country.

Two more five-part decisions turn these raw sources into the analytic
displacement layer:

| # | Decision | Question |
|---|----------|----------|
| 7 | DTM operation selection | DTM ships three overlapping IDP series — which one? |
| 5 | DTM ↔ UNHCR reconciliation | How do the internal and cross-border sources combine? |

Decision 7 comes first: we cannot reconcile DTM against UNHCR (Decision 5)
until we have settled which DTM series *is* the internal layer.

### Decision 7: Which DTM operation is the internal-displacement layer

**Problem.** The pinned DTM snapshot is not one series. The `operation` column
holds **three** of them — `Armed Clashes in Sudan` (a weekly tracker), the same
`(Monthly)`, and `(Overview)` — and they all report `numPresentIdpInd` for the
*same* IDP population. They are DTM products, not independent measurements:
stacking them would triple-count displacement, and even picking the wrong single
one distorts both the magnitude and the time span of every displacement figure
downstream. The choice is load-bearing, so it is explicit.

**Diagnostic.** National present-IDP stock (latest round in each month) under
each operation, plus each operation's date span.

In [8]:
dtm = pd.read_parquet(data._latest_snapshot("dtm_admin1_snapshot"))
dtm["reportingDate"] = pd.to_datetime(dtm["reportingDate"])
dtm["ym"] = dtm["reportingDate"].dt.to_period("M")

# Date span + national IDP stock at the first and last month of each operation.
rows = []
for op, g in dtm.groupby("operation"):
    last_in_month = g.loc[g.groupby("ym")["reportingDate"].transform("max")
                          == g["reportingDate"]]
    monthly = last_in_month.groupby("ym")["numPresentIdpInd"].sum()
    rows.append({
        "operation": op.replace("Armed Clashes in Sudan", "ACiS"),
        "first_month": str(monthly.index.min()),
        "last_month": str(monthly.index.max()),
        "n_months": monthly.index.nunique(),
        "idp_first_month": int(monthly.iloc[0]),
        "idp_last_month": int(monthly.iloc[-1]),
    })
op_summary = pd.DataFrame(rows)
display(op_summary)

# The operations disagree on the *same* month — they are not stitchable.
overlap = ["2023-12", "2024-04"]
clash = {}
for op, g in dtm.groupby("operation"):
    last_in_month = g.loc[g.groupby("ym")["reportingDate"].transform("max")
                          == g["reportingDate"]]
    m = last_in_month.groupby("ym")["numPresentIdpInd"].sum()
    clash[op.replace("Armed Clashes in Sudan", "ACiS")] = {
        p: int(m[p]) if pd.Period(p) in m.index else None for p in overlap}
print("National IDP stock reported for the SAME month, by operation:")
display(pd.DataFrame(clash).T)

,operation,first_month,last_month,n_months,idp_first_month,idp_last_month
0,ACiS,2023-04,2024-04,13,334053,6720136
1,ACiS (Monthly),2023-09,2024-04,8,4295101,6786816
2,ACiS (Overview),2023-08,2026-02,23,7071674,9044786


National IDP stock reported for the SAME month, by operation:


,2023-12,2024-04
ACiS,5942580.0,6720136.0
ACiS (Monthly),5856777.0,6786816.0
ACiS (Overview),9052822.0,NaN


**Options considered.**
- (a) **`ACiS` weekly** — finest cadence (45 rounds), but stops at Apr 2024 and
  its national total only ramps to 6.7M: it is an *assessment-coverage* curve,
  not a displacement curve, growing as DTM reached more areas.
- (b) **`ACiS (Monthly)`** — clean monthly cadence but only 8 months
  (Sep 2023 – Apr 2024); far too short for a two-year analysis.
- (c) **`ACiS (Overview)`** — the consolidated series DTM headlines: Aug 2023 →
  Feb 2026, 23 reporting rounds, with full admin-1 origin breakdowns.

**Decision.** Option (c): **`Armed Clashes in Sudan (Overview)`** is the
internal-displacement layer. It is the only operation that spans the analytic
window, and the diagnostic shows the three series are *incompatible*, not
complementary — for Dec 2023 the weekly tracker reports ~5.9M IDPs while the
Overview reports ~9.1M, because they consolidate field assessments differently.
Stitching them would inject a spurious step; picking the short ones truncates
the analysis. `(Overview)` is encoded as `data.DTM_OPERATION`.

**Sensitivity.** This is a source-selection choice, so the layer's *level*
depends on it — but the **geographic and temporal shape does not**: the three
operations rank the Darfur states top and the eastern states bottom identically,
and all three trace the same rising arc through 2023–24. The hero finding is the
co-evolution of violence and displacement *geography*, which is robust to the
operation chosen. The Apr–Jul 2023 gap (the Overview starts in August, four
months after war onset) is a real coverage limit, carried to the README.

### Decision 5: Reconciling the internal (DTM) and cross-border (UNHCR) sources

**Problem.** The project brief anticipated a head-to-head reconciliation —
"where IOM DTM and UNHCR both report the same Sudan → neighbour flow, choose one
as primary." The diagnostic below shows that framing does not fit the data:
**DTM and UNHCR never measure the same flow.** DTM counts IDPs *inside* Sudan;
UNHCR counts refugees who *left* it. They are two halves of the displacement
picture, not two estimates of one number — summing them into a single
"displacement" total would be a category error. The real disagreement is
*within* UNHCR: its two public sources report the cross-border flow at very
different magnitudes, and that gap must be resolved before the chord diagram.

**Diagnostic.** First, confirm DTM and UNHCR partition the flow space. Then
compare the two UNHCR cross-border sources — the Refugee Statistics API
(legal-status counts) and the situation portal snapshot (operational arrival
estimates) — per destination.

In [9]:
# (1) DTM and UNHCR cover disjoint flows — internal vs cross-border.
print("IOM DTM   — flow measured:", "Sudan admin-1  ->  Sudan admin-1   (internal IDPs)")
print("UNHCR     — flow measured:", "Sudan          ->  destination country (refugees)")
print("Shared origin-destination pairs:", 0, "— nothing to reconcile head-to-head.\n")

# (2) The genuine conflict: UNHCR's two sources, per destination country.
stats = data.fetch_unhcr_statistics()
stats_latest = stats[stats["year"] == stats["year"].max()]
portal = data.fetch_unhcr_situation_countries()

name_fix = {"Central African Rep.": "Central African Republic"}
s = (stats_latest.assign(coa_name=stats_latest["coa_name"].replace(name_fix))
     .set_index("coa_name"))
unhcr_cmp = (portal.assign(
        portal_snapshot=portal["individuals"],
        stats_refugees=portal["destination"].map(s["refugees"]),
        stats_asylum_seekers=portal["destination"].map(s["asylum_seekers"]),
    )[["destination", "portal_snapshot", "stats_refugees", "stats_asylum_seekers"]])
unhcr_cmp["portal / stats_refugees"] = (
    unhcr_cmp["portal_snapshot"] / unhcr_cmp["stats_refugees"]).round(1)
display(unhcr_cmp)

print("Destination ranking by portal snapshot :",
      list(unhcr_cmp.sort_values("portal_snapshot", ascending=False)["destination"]))
print("Destination ranking by stats refugees  :",
      list(unhcr_cmp.sort_values("stats_refugees", ascending=False)["destination"]))

IOM DTM   — flow measured: Sudan admin-1  ->  Sudan admin-1   (internal IDPs)
UNHCR     — flow measured: Sudan          ->  destination country (refugees)
Shared origin-destination pairs: 0 — nothing to reconcile head-to-head.



,destination,portal_snapshot,stats_refugees,stats_asylum_seekers,portal / stats_refugees
0,Egypt,1500000,31366,706027,47.8
1,Chad,926963,1258904,3241,0.7
2,Libya,559920,314770,71388,1.8
3,South Sudan,448219,551284,12,0.8
4,Uganda,89924,85179,205,1.1
5,Ethiopia,55826,96167,438,0.6
6,Central African Republic,36342,38855,105,0.9


Destination ranking by portal snapshot : ['Egypt', 'Chad', 'Libya', 'South Sudan', 'Uganda', 'Ethiopia', 'Central African Republic']
Destination ranking by stats refugees  : ['Chad', 'South Sudan', 'Libya', 'Ethiopia', 'Uganda', 'Central African Republic', 'Egypt']


**Options considered.**
- (a) **Reconcile DTM against UNHCR into one displacement number** — rejected: the
  diagnostic shows they measure disjoint flows (internal vs cross-border); there
  is no shared origin-destination pair, so a "reconciled" figure would be
  meaningless or a double-count.
- (b) **UNHCR Refugee Statistics API** as the cross-border source — clean annual
  series, but it counts *legal status*: most Sudanese in Egypt are asylum-seekers
  or unregistered, so it reads Egypt at ~31k refugees and ranks it dead last.
- (c) **UNHCR situation portal snapshot** as the cross-border source —
  operational total-presence estimates (~1.5M for Egypt), at the cost of
  per-country `as_of_date`s that differ.

**Decision.** Keep **two complementary layers, never summed**: DTM is the
internal layer, UNHCR the cross-border layer. For the cross-border layer the
primary source is the **situation portal snapshot** (option c) — it is the
total-presence figure UNHCR itself headlines, and it is the methodological twin
of DTM's "present IDP" stock (a point-in-time *presence* count, not a
legal-status count), which keeps the two layers comparable. The Statistics API
is retained in the notebook as the registered-refugee lower bound.

**Sensitivity.** The cross-border *magnitude* and even the destination *ranking*
are sensitive to this choice: Egypt is the **#1** destination under the portal
snapshot but **last of the seven** under registered-refugee counts (a 48× gap),
because Sudanese in Egypt are largely unregistered. This is not hidden — the
comparison table above is in the notebook, and the README states which figure
each chart uses. The internal-vs-cross-border *split*, the headline of this
layer, does not depend on the choice at all.

### Building the displacement layer

The two decisions are encoded as the defaults of `build_displacement_layer()`
and `build_displacement_od()`. The first produces the **internal IDP layer** —
present-IDP stock per Sudan state × month, the displacement twin of the violence
layer — from the `(Overview)` operation (D7). Because DTM assesses on an
irregular cadence and `numPresentIdpInd` is a *stock* (it persists between
rounds), missing months are **forward-filled**, not zero-filled — the opposite
of the violence layer, where a missing month genuinely means zero events.

The second produces the **origin-destination dataset** for the chord diagram:
internal admin-1 → admin-1 flows from DTM, plus cross-border Sudan → country
flows from UNHCR (D5), in one tidy frame.

In [10]:
displacement = data.build_displacement_layer()   # D7 default; pins the parquet
od = data.build_displacement_od()                # D5/D7 defaults; pins the parquet

n_states = displacement["canonical_pcode"].nunique()
n_months = displacement["period"].nunique()
# How many of those months had no DTM assessment round and inherit the prior
# stock by forward-fill (D7)?
_lo, _hi = (pd.Timestamp(d) for d in data.DISPLACEMENT_WINDOW)
_ov = dtm[dtm["operation"] == data.DTM_OPERATION]
n_round_months = _ov[(_ov["reportingDate"] >= _lo)
                     & (_ov["reportingDate"] <= _hi)]["ym"].nunique()

print(f"Internal IDP layer : {displacement.shape[0]} rows = "
      f"{n_states} states x {n_months} months "
      f"({displacement['period'].min():%Y-%m} -> {displacement['period'].max():%Y-%m})")
print(f"Forward-filled gaps: {n_months - n_round_months} of {n_months} months had no "
      f"DTM round and inherit the prior stock")
peak = displacement.groupby("period")["idp_present"].sum()
print(f"National IDP stock : {peak.iloc[0]:,} (Aug 2023) -> peak "
      f"{peak.max():,} ({peak.idxmax():%b %Y}) -> {peak.iloc[-1]:,} (May 2025)")
print()
print(f"O-D dataset        : {len(od)} flows  "
      f"({(od.flow_type == 'internal').sum()} internal + "
      f"{(od.flow_type == 'cross_border').sum()} cross-border)")
print("Pinned to          : data/processed/displacement_admin1_monthly.parquet")
print("                     data/processed/displacement_od.parquet")
displacement.head()

Internal IDP layer : 396 rows = 18 states x 22 months (2023-08 -> 2025-05)
Forward-filled gaps: 8 of 22 months had no DTM round and inherit the prior stock
National IDP stock : 7,071,674 (Aug 2023) -> peak 11,585,384 (Jan 2025) -> 10,136,005 (May 2025)

O-D dataset        : 231 flows  (224 internal + 7 cross-border)
Pinned to          : data/processed/displacement_admin1_monthly.parquet
                     data/processed/displacement_od.parquet


,canonical_pcode,admin1,period,idp_present
0,SD01,Khartoum,2023-08-01,40800
1,SD01,Khartoum,2023-09-01,40800
2,SD01,Khartoum,2023-10-01,40800
3,SD01,Khartoum,2023-11-01,40800
4,SD01,Khartoum,2023-12-01,37870


In [11]:
# The geographic shape of internal displacement — IDP stock at the latest
# in-window month, beside the cross-border destinations.
latest = displacement[displacement["period"] == displacement["period"].max()]
print(f"Internal IDPs by host state — {displacement['period'].max():%B %Y}")
display(latest.sort_values("idp_present", ascending=False)
        [["admin1", "idp_present"]].reset_index(drop=True))

print("\nCross-border refugees by destination — UNHCR portal snapshot (Decision 5)")
display(od[od["flow_type"] == "cross_border"]
        .sort_values("individuals", ascending=False)
        [["destination", "individuals", "as_of_date"]].reset_index(drop=True))

Internal IDPs by host state — May 2025


,admin1,idp_present
0,South Darfur,1842508
1,North Darfur,1793938
2,Central Darfur,949647
3,East Darfur,802844
4,River Nile,619654
5,White Nile,563879
6,Gedaref,533899
7,Northern,532656
8,Blue Nile,400971
9,West Kordofan,393814



Cross-border refugees by destination — UNHCR portal snapshot (Decision 5)


,destination,individuals,as_of_date
0,Egypt,1500000,2025-01-31
1,Chad,926963,2026-05-04
2,Libya,559920,2026-04-20
3,South Sudan,448219,2026-05-04
4,Uganda,89924,2026-05-04
5,Ethiopia,55826,2025-09-08
6,Central African Republic,36342,2026-04-20


The displacement layer is built and pinned. The **internal layer** —
`data/processed/displacement_admin1_monthly.parquet`, 396 rows (18 states ×
22 months) — shows the IDP stock rising from 7.1M (Aug 2023) to a peak of 11.6M
(Jan 2025), then easing as early returns begin; the **Darfur states host the
largest IDP populations**, the same region the violence layer lights up. The
**O-D dataset** — `displacement_od.parquet` — carries 224 internal admin-1
flows plus 7 cross-border destinations, feeding the chord diagram in the next
section.

> **Note — the two layers do not share a start date.** The violence layer opens
> in April 2023; the DTM `(Overview)` series begins in **August 2023** (D7), so
> co-registered figures cover the war's first four months on the violence axis
> only. The cross-border snapshot dates differ per country (D5). Both gaps are
> recorded in the README limitations.